In [1]:
# Setting up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

In [ ]:
# Importing packages and modules
import comet_ml
from openpyxl import Workbook, load_workbook
from datetime import datetime
import torch
from matplotlib import pyplot as plt
from pytorch_lightning.loggers import CometLogger
from copy import deepcopy
import numpy as np
import gymnasium as gym
from itertools import product
from tqdm import tqdm
from RL4CRN.Input_Output_Rxn_Networks.IOCRN_MassAction import IOCRN_MassAction
from RL4CRN.Environments.CRNEnvironment import CRNEnvironment
from RL4CRN.Environments.VecCRNEnvironment import VecCRNEnvironment
from RL4CRN.Environments.VecCRNEnvironment import SerialVecCRNEnvironment
from RL4CRN.Agents.RecurrentAgent import RecurrentAgent
from RL4CRN.Input_Output_Rxn_Networks.CRNGenerator import CRNCompletor
from RL4CRN.Rewards.Structure import compute_stoichiometry_rank

In [ ]:
# Set the logger to use Comet
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
api_key = "o77J6VCMDamustkfJuMXZ2jdV"
logger = CometLogger(
    api_key=api_key,
    project="Rank",        
    workspace="maurice-filo", 
    experiment_name=f'Rank_Experiment_{timestamp}',
)
logger = logger.experiment

In [ ]:
# Construct the template CRN
species_labels = ['X_1', 'X_2', 'X_3', 'X_4', 'X_5', 'X_6']
inputs_labels = ['u_1']
stoichiometry_reactants = np.array([ [] for _ in species_labels], dtype=np.int8)
stoichiometry_products = np.array([ [] for _ in species_labels], dtype=np.int8)
parameters = np.array([], dtype=np.float32)
input_influence_matrix = np.array([[] for _ in inputs_labels], dtype=np.int8)
outputs = np.array([1], dtype=np.int8)
CRN_template = IOCRN_MassAction(stoichiometry_reactants, stoichiometry_products, parameters, input_influence_matrix, outputs, species_labels, inputs_labels)
print('CRN template:')
CRN_template.print_reactions()

In [ ]:
# Hyperparameters
max_num_reactions = 20                              # Maximum number of reactions
N_CPUs = 128//32                                        # Number of CPUs          
n_samples = 10*32*N_CPUs                               # Number of samples    
n_grid = 1
param_lower_bound = 1
param_upper_bound = 1
width = 128
depth = 5
num_species = 6
num_inputs = 1
allow_input_influence = False
learning_rate = 1e-3
entropy_weight = 0
entropy_update_coefficient = 0
entropy_schedule = 1000
minimum_entropy_weight = 0
risk = 0
risk_update = 0
maximum_risk = 1.0
risk_schedule = 1000
epoch_num = 200
render_schedule = 1
t_f = 0
mode = {'style': 'logger', 'task': 'rank', 'format': 'figure'}

# Create the reward function
def compute_reward(state):
    target_rank = 4
    return compute_stoichiometry_rank(state, target_rank)

# Sheet File
file_name = "Rank.xlsx"

In [ ]:
sheet_name = "Data"
headers = ["Timestamp", "URL", "Maximum Number of Reactions", "Number of Species", "Number of Inputs", "Number of Samples", "Final Time", 
           "Allow Input Influence", "Number of Grid Points", "Parameter Lower Bound", "Parameter Upper Bound",
           "Learning Rate", "Initial Entropy Weight", "Entropy Update Coefficient", "Entropy Schedule",
           "Minimum Entropy Weight", "Risk", "Risk Update", "Maximum Risk", "Risk Schedule",
           "Number of Epochs", "Render Schedule", "Neural Network Depth", "Neural Network Width", "Number of CPUs"]
data_row = [timestamp, None if logger is None else logger.url, max_num_reactions, num_species, num_inputs, n_samples, t_f,
            allow_input_influence, n_grid, param_lower_bound, param_upper_bound, 
            learning_rate, entropy_weight, entropy_update_coefficient, entropy_schedule, 
            minimum_entropy_weight, risk, risk_update, maximum_risk, risk_schedule,
            epoch_num, render_schedule, depth, width, N_CPUs]

if os.path.exists(file_name):
    wb = load_workbook(file_name)
    if sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
    else:
        ws = wb.create_sheet(sheet_name)
else:
    wb = Workbook()
    ws = wb.active
    ws.title = sheet_name

if ws.max_column < 2:
    for i, header in enumerate(headers, start=1):
        ws.cell(row=i, column=1, value=header)
next_col = ws.max_column + 1
for i, value in enumerate(data_row, start=1):
    ws.cell(row=i, column=next_col, value=value)

wb.save(file_name)
print(f"New experiment data saved in column {next_col} of '{file_name}'.")

In [ ]:
# Construct parallel environments
CRN_0 = deepcopy(CRN_template)
vec_env = VecCRNEnvironment([CRNEnvironment(CRN_0, max_num_reactions, logger=logger, logger_schedule=1) for _ in range(n_samples)], N_CPUs=N_CPUs, logger=logger)
# vec_env = SerialVecCRNEnvironment([CRNEnvironment(CRN_0, max_num_reactions, logger=logger, logger_schedule=1) for _ in range(n_samples)], logger=logger)

In [ ]:
# Construct the agent
device = 'cuda' if torch.cuda.is_available() else 'cpu'
param_grid = np.linspace(param_lower_bound, param_upper_bound, n_grid, dtype=np.float32)
parameter_grid = np.tile(param_grid, (max_num_reactions, 1))
class RSG_Attributes:
    def __init__(self, width, depth, allow_input_influence):
        self.LSTM_hidden_size = width
        self.FFNN_hidden_size = [width, width, width] if allow_input_influence else [width, width]
        self.FFNN_num_layers = [depth, depth, depth] if allow_input_influence else [depth, depth]
        self.weight = [None, None, None] if allow_input_influence else [None, None]
class PSG_Attributes:
    def __init__(self, width, depth):
        self.LSTM_hidden_size = width
        self.FFNN_hidden_size = width
        self.FFNN_num_layers = 3
        self.weight = None

rsg_attributes = RSG_Attributes(width, depth, allow_input_influence)
psg_attributes = PSG_Attributes(width, depth)
completor = CRNCompletor(max_num_reactions, num_species, max_num_reactions, CRN_0.num_unknown_parameters, num_inputs, parameter_grid, n_samples, rsg_attributes, psg_attributes, device=device, allow_input_influence=allow_input_influence).to(device)
agent = RecurrentAgent(vec_env.envs[0], completor, allow_input_influence, logger, learning_rate, entropy_weight, entropy_update_coefficient, entropy_schedule, minimum_entropy_weight, risk, risk_update, maximum_risk, risk_schedule)

In [ ]:
# Training Loop
for i in tqdm(range(epoch_num)):
    vec_env.reset()
    for j in range(max_num_reactions + vec_env.envs[0].CRN_template.num_unknown_parameters):
        actions = agent.act()
        out = vec_env.step(actions)
    rewards = vec_env.get_reward(compute_reward)
    agent.update(rewards)
    if i % render_schedule == 0:
        vec_env.render(rewards, mode=mode)